## Setup

This notebook generates 3D visualizations of semantic trajectories in embedding space using UMAP dimensionality reduction.

In [ ]:
import sys
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import torch
import umap

sys.path.append("../src")

import utils
import mappings

# Reload modules
importlib.reload(utils)
importlib.reload(mappings)

%matplotlib inline

In [ ]:
# Available datasets
DATASETS = [
    "parkinson",  # Neurodegenerative (Spanish)
    "swear-fluency",  # Swear Fluency (English)
    "italian",  # Italian Property Listing
    "german",  # German Property Listing
]

# Available embedding backends
BACKENDS = [
    "openai",  # OpenAI text-embedding-3-large
    "gemini",  # Google text-embedding-004
    "qwen",  # Qwen-Embedding-0.6B
    "fasttext",  # FastText
    # "word2vec",  # Word2Vec (optional)
]

# Output directory for visualizations
OUTPUT_DIR = Path("../results/3dviz")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datasets: {DATASETS}")
print(f"Backends: {BACKENDS}")
print(f"Output directory: {OUTPUT_DIR}")

## Utility Functions

In [ ]:
def reduce_to_3d(embeddings, n_neighbors=15, min_dist=0.0, random_state=42):
    """Reduce high-dimensional embeddings to 3D using UMAP.

    Args:
        embeddings: High-dimensional embeddings array of shape (n_samples, n_features).
        n_neighbors: Number of neighbors for UMAP (auto-adjusted based on sample size).
            Default is 15.
        min_dist: Minimum distance parameter for UMAP. Default is 0.0.
        random_state: Random seed for reproducibility. Default is 42.

    Returns:
        3D coordinates array of shape (n_samples, 3).
    """
    n_samples = embeddings.shape[0]
    # Adjust n_neighbors based on sample size (must be < n_samples)
    n_neighbors = min(n_neighbors, max(3, int(0.2 * n_samples)))

    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="cosine",
        random_state=random_state,
    )

    return reducer.fit_transform(embeddings)


def add_axis_triad(ax, center, scale=0.1, linewidth=2.0):
    """Add small RGB axis triad to show orientation in 3D space.

    Args:
        ax: Matplotlib 3D axis object.
        center: Center position as array-like (x, y, z).
        scale: Length of axis arrows. Default is 0.1.
        linewidth: Width of axis lines. Default is 2.0.
    """
    x0, y0, z0 = center
    axes = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])  # x, y, z directions
    colors = ["red", "green", "blue"]  # RGB

    for direction, color in zip(axes, colors):
        ax.quiver(
            x0,
            y0,
            z0,
            direction[0],
            direction[1],
            direction[2],
            color=color,
            length=scale,
            linewidth=linewidth,
            arrow_length_ratio=0.3,
        )


def plot_colored_trajectory(ax, xyz, timesteps, cmap="coolwarm", linewidth=2.5):
    """Plot a 3D trajectory with color-coded timesteps.

    Args:
        ax: Matplotlib 3D axis object.
        xyz: 3D coordinates array of shape (n_points, 3).
        timesteps: Time indices array for coloring.
        cmap: Colormap name. Default is "coolwarm".
        linewidth: Width of trajectory line. Default is 2.5.

    Returns:
        Line3DCollection object for use with colorbar.
    """
    # Create line segments
    points = xyz.reshape(-1, 1, 3)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    # Create colored line collection
    lc = Line3DCollection(
        segments, array=timesteps[:-1], cmap=cmap, linewidth=linewidth
    )
    ax.add_collection3d(lc)

    # Set axis limits
    ax.set_xlim(xyz[:, 0].min(), xyz[:, 0].max())
    ax.set_ylim(xyz[:, 1].min(), xyz[:, 1].max())
    ax.set_zlim(xyz[:, 2].min(), xyz[:, 2].max())

    # Clean appearance
    ax.set_axis_off()

    # Add axis triad at trajectory centroid
    center = xyz.mean(axis=0)
    add_axis_triad(ax, center, scale=0.15)

    return lc


def load_embeddings(dataset, backend):
    """Load embedding data for a specific dataset and backend.

    Args:
        dataset: Dataset name (e.g., "parkinson").
        backend: Backend name (e.g., "openai").

    Returns:
        DataFrame containing loaded embeddings with metadata.

    Raises:
        FileNotFoundError: If embeddings file does not exist.
    """
    embedding_path = f"../data/embeddings/{backend}/{dataset}.csv"

    if not os.path.exists(embedding_path):
        raise FileNotFoundError(f"Embeddings not found: {embedding_path}")

    df = utils.load(embedding_path)
    print(
        f"Loaded {dataset} ({backend}): {df.shape[0]} rows, {df['id'].nunique()} participants"
    )

    return df


def get_example_concept(df, dataset):
    """Get a representative concept for visualization from a dataset.

    Args:
        df: DataFrame containing embedding data.
        dataset: Dataset name.

    Returns:
        Concept name as string. Returns preferred concept if available,
        otherwise returns first concept in the dataframe.
    """
    # Dataset-specific concept selection
    concept_preferences = {
        "parkinson": "TIBURON",  # Shark - common concept
        "swear-fluency": "ANIMAL",  # Animal category
        "italian": "mano",  # Hand
        "german": "Besen",  # Broom
    }

    preferred = concept_preferences.get(dataset)

    # Check if preferred concept exists
    if preferred and preferred in df["concept"].values:
        return preferred

    # Otherwise, return first concept
    return df["concept"].iloc[0]

## Main Visualization Function

In [ ]:
def visualize_trajectories(
    df,
    n_examples=5,
    concept=None,
    figsize=(10, 15),
    cmap="coolwarm",
    save_path=None,
):
    """Create a grid visualization of semantic trajectories.

    Generates a 2-column grid showing cumulative vs. non-cumulative embeddings
    for multiple participants' semantic trajectories in 3D space.

    Args:
        df: DataFrame with 'embedding' and 'prop_embedding' columns containing
            the trajectory data.
        n_examples: Number of participant examples to visualize. Default is 5.
        concept: Specific concept to visualize. If None, uses first available
            concept. Default is None.
        figsize: Figure size as tuple (width, height). Default is (10, 15).
        cmap: Colormap name for trajectory coloring. Default is "coolwarm".
        save_path: Path to save figure. If None, displays only without saving.
            Default is None.

    Returns:
        Matplotlib Figure object containing the visualization.
    """
    # Select concept
    if concept is None:
        concept = df["concept"].iloc[0]

    # Get participant IDs
    participant_ids = df["id"].unique()[:n_examples]
    n_rows = len(participant_ids)

    # Create figure with 3D subplots (2 columns: cumulative vs. non-cumulative)
    fig, axes = plt.subplots(
        n_rows,
        2,
        subplot_kw={"projection": "3d"},
        figsize=figsize,
    )
    fig.patch.set_facecolor("white")

    # Ensure axes is always 2D array
    if n_rows == 1:
        axes = np.array([axes])

    # Store last line collections for colorbar
    last_lc_cumulative = None
    last_lc_noncumulative = None

    # Plot each participant
    for row, participant_id in enumerate(participant_ids):
        ax_cumulative = axes[row, 0]
        ax_noncumulative = axes[row, 1]

        # Filter data for this participant and concept
        trajectory = df[(df["id"] == participant_id) & (df["concept"] == concept)]

        if trajectory.empty:
            ax_cumulative.set_axis_off()
            ax_noncumulative.set_axis_off()
            continue

        print(f"  Participant {participant_id}: {len(trajectory)} timesteps")

        # Extract embeddings
        cumulative_emb = torch.stack(trajectory["embedding"].tolist()).cpu().numpy()
        noncumulative_emb = (
            torch.stack(trajectory["prop_embedding"].tolist()).cpu().numpy()
        )

        # Reduce to 3D
        cumulative_3d = reduce_to_3d(cumulative_emb)
        noncumulative_3d = reduce_to_3d(noncumulative_emb)

        # Timesteps for coloring
        timesteps = np.arange(len(trajectory))

        # Plot trajectories
        lc_cumulative = plot_colored_trajectory(
            ax_cumulative, cumulative_3d, timesteps, cmap=cmap
        )
        lc_noncumulative = plot_colored_trajectory(
            ax_noncumulative, noncumulative_3d, timesteps, cmap=cmap
        )

        last_lc_cumulative = lc_cumulative
        last_lc_noncumulative = lc_noncumulative

        # Add column titles on first row
        if row == 0:
            ax_cumulative.set_title("Cumulative Embeddings", fontsize=12, pad=10)
            ax_noncumulative.set_title("Non-Cumulative Embeddings", fontsize=12, pad=10)

        # Add row label
        ax_cumulative.text2D(
            -0.05,
            0.5,
            f"ID: {participant_id}",
            transform=ax_cumulative.transAxes,
            fontsize=10,
            rotation=90,
            va="center",
            ha="right",
        )

    # Add colorbars
    if last_lc_cumulative is not None:
        cbar_cumulative = fig.colorbar(
            last_lc_cumulative,
            ax=axes[:, 0].ravel().tolist(),
            fraction=0.015,
            pad=0.02,
        )
        cbar_cumulative.set_label("Timestep", fontsize=10)

    if last_lc_noncumulative is not None:
        cbar_noncumulative = fig.colorbar(
            last_lc_noncumulative,
            ax=axes[:, 1].ravel().tolist(),
            fraction=0.015,
            pad=0.02,
        )
        cbar_noncumulative.set_label("Timestep", fontsize=10)

    # Add main title
    fig.suptitle(f"Semantic Trajectories for Concept: {concept}", fontsize=14, y=0.995)

    plt.tight_layout()

    # Save if path provided
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved to: {save_path}")

    plt.show()

    return fig

## Single Dataset Visualization

Visualize trajectories for a specific dataset and backend.

In [ ]:
# Configuration for single visualization
DATASET = "parkinson"  # Change to desired dataset
BACKEND = "openai"  # Change to desired backend
N_EXAMPLES = 5  # Number of participants to show

print(f"Visualizing: {DATASET} with {BACKEND}\n")

# Load data
df = load_embeddings(DATASET, BACKEND)

# Get representative concept
concept = get_example_concept(df, DATASET)
print(f"Selected concept: {concept}\n")

# Create visualization
save_path = OUTPUT_DIR / f"{DATASET}_{BACKEND}_{concept}.png"
visualize_trajectories(
    df,
    n_examples=N_EXAMPLES,
    concept=concept,
    figsize=(10, 3 * N_EXAMPLES),
    save_path=save_path,
)

## Batch Processing: All Datasets and Backends

Generate visualizations for all combinations of datasets and backends.

In [ ]:
def batch_visualize_all(datasets, backends, n_examples=5, output_dir=OUTPUT_DIR):
    """Generate visualizations for all dataset-backend combinations.

    Processes all combinations of datasets and backends, creating and saving
    3D trajectory visualizations for each.

    Args:
        datasets: List of dataset names to process.
        backends: List of backend names to process.
        n_examples: Number of participant examples per visualization. Default is 5.
        output_dir: Directory path to save figures. Default is OUTPUT_DIR.
    """
    total = len(datasets) * len(backends)
    current = 0

    print(f"Generating {total} visualizations...\n")

    for dataset in datasets:
        for backend in backends:
            current += 1
            print(f"[{current}/{total}] {dataset} + {backend}")

            try:
                # Load embeddings
                df = load_embeddings(dataset, backend)

                # Get concept
                concept = get_example_concept(df, dataset)
                print(f"  Concept: {concept}")

                # Create output path
                save_path = output_dir / f"{dataset}_{backend}_{concept}.png"

                # Visualize
                visualize_trajectories(
                    df,
                    n_examples=n_examples,
                    concept=concept,
                    figsize=(10, 3 * n_examples),
                    save_path=save_path,
                )

                plt.close()  # Close figure to free memory

            except FileNotFoundError as e:
                print(f"  Skipped: {e}")
            except Exception as e:
                print(f"  Error: {e}")

            print()

    print(f"Completed! All visualizations saved to: {output_dir}")

In [ ]:
# Run batch processing
# WARNING: This will generate many figures and may take some time

batch_visualize_all(
    datasets=DATASETS,
    backends=BACKENDS,
    n_examples=5,
    output_dir=OUTPUT_DIR,
)

## Backend Comparison for Single Dataset

Compare how different embedding models represent the same trajectories.

In [ ]:
def compare_backends(dataset, backends, n_examples=3, concept=None):
    """Compare trajectory visualizations across different backends for one dataset.

    Creates visualizations for the same dataset using different embedding backends
    to compare how different models represent semantic trajectories.

    Args:
        dataset: Dataset name to visualize.
        backends: List of backend names to compare.
        n_examples: Number of participant examples to show. Default is 3.
        concept: Specific concept to visualize. If None, automatically selected
            based on dataset. Default is None.
    """
    print(f"Comparing backends for {dataset}\n")

    for backend in backends:
        print(f"Backend: {backend}")

        try:
            df = load_embeddings(dataset, backend)

            if concept is None:
                concept = get_example_concept(df, dataset)

            save_path = OUTPUT_DIR / f"compare_{dataset}_{backend}_{concept}.png"

            visualize_trajectories(
                df,
                n_examples=n_examples,
                concept=concept,
                figsize=(10, 3 * n_examples),
                save_path=save_path,
            )

            plt.close()

        except Exception as e:
            print(f"  Error: {e}")

        print()


# Example: Compare all backends for Parkinson dataset
compare_backends(
    dataset="parkinson",
    backends=BACKENDS,
    n_examples=3,
)

## Custom Visualization

Create a custom visualization with specific parameters.

In [ ]:
# Custom parameters
CUSTOM_DATASET = "italian"
CUSTOM_BACKEND = "openai"
CUSTOM_CONCEPT = "mano"  # Hand in Italian
CUSTOM_N_EXAMPLES = 6
CUSTOM_CMAP = "viridis"  # Try: 'plasma', 'inferno', 'coolwarm', 'twilight'

# Load and visualize
df_custom = load_embeddings(CUSTOM_DATASET, CUSTOM_BACKEND)

visualize_trajectories(
    df_custom,
    n_examples=CUSTOM_N_EXAMPLES,
    concept=CUSTOM_CONCEPT,
    figsize=(10, 3 * CUSTOM_N_EXAMPLES),
    cmap=CUSTOM_CMAP,
    save_path=OUTPUT_DIR
    / f"custom_{CUSTOM_DATASET}_{CUSTOM_BACKEND}_{CUSTOM_CONCEPT}.png",
)